# Индекс корпуса

Переводим документы в векторы заранее, и кладём файлом рядом
с BLS. В рантайме Triton считает вектор только для вопроса пользователя,
а документы уже готовы.

На выходе два файла:

| Файл | Что внутри |
|---|---|
| `corpus/documents.json` | тексты и названия, ~2 МБ |
| `corpus/vectors.npy` | матрица [509, 384], ~0.8 МБ |

Поиск в BLS будет одной строкой: `vectors @ query_vector` — умножили
матрицу на вектор, взяли три наибольших числа.

In [9]:
import os
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

CORPUS_PATH = Path("data/search_documents.csv")
OUTPUT_DIR = Path("model_repository/assistant_bls/1/corpus")

MODEL_NAME = "intfloat/multilingual-e5-small"

# короче 500 символов текст - не берем
MIN_CONTEXT_LENGTH = 150

# самый длинный search_text корпуса — 277 токенов, 512 хватает.
INDEX_MAX_LENGTH = 512

# сколько символов описания уйдёт генератору. Три документа по 900 символов
# это примерно 900 токенов, плюс вопрос и ответ — влезает в max_model_len 2048
MAX_CONTEXT_CHARS = 900

print("Рабочая папка:", Path.cwd())
print("Корпус на месте:", CORPUS_PATH.exists())

Рабочая папка: c:\devs\AI\LlmEngineer\llm-eng-26-triton
Корпус на месте: True


## 1. Что в корпусе

Две текстовые колонки, и у каждой своя работа:

| Колонка | Для чего |
|---|---|
| `search_text` | **для поиска.** Короткая: название, город, категория, адрес. Её переводим в вектор |
| `context_text` | **для ответа.** Развёрнутое описание. Её отдаём генератору |

По короткому тексту ищем. Длинный даёт генератору факты.

In [10]:
import csv

with CORPUS_PATH.open(encoding="utf-8") as handle:
    rows = list(csv.DictReader(handle))

print("Строк:", len(rows))
print("Колонки:", list(rows[0].keys()))
print()

example = rows[0]
print("search_text :", example["search_text"][:200])
print()
print("context_text:", example["context_text"][:400])

Строк: 2686
Колонки: ['document_id', 'document_kind', 'place_id', 'category', 'name', 'city', 'latitude', 'longitude', 'search_text', 'context_text', 'source_ids']

search_text : Lady Stretch — объект в городе Ессентуки. Часы работы по данным карты: Mo-Su 09:00-14:00,16:00-21:00. Сайт: https://ladystretch.com/essentuki. Телефон: +7 938 3044110.

context_text: Lady Stretch — объект в городе Ессентуки. Часы работы по данным карты: Mo-Su 09:00-14:00,16:00-21:00. Сайт: https://ladystretch.com/essentuki. Телефон: +7 938 3044110.


## 2. Порог отбора

У части объектов `context_text` почти совпадает с `search_text` — это карточки
вида «Разгонин А.И. — арт-объект в городе Минеральные Воды». Генератору
по ним отвечать нечем, в индекс они не нужны.

Берём порог **500 символов**. Останется около 509 документов с медианой
описания ~1825 символов — то есть каждый отобранный документ действительно
содержит текст, по которому можно ответить.

In [11]:
import numpy as np

lengths = np.array([len(row["context_text"]) for row in rows])

print("Длина context_text по всему файлу:")
for percent in [50, 75, 90, 100]:
    print(f"  перцентиль {percent:3}: {np.percentile(lengths, percent):6.0f} символов")
print()

for threshold in [150, 300, 500, 800]:
    kept_count = int((lengths >= threshold).sum())
    median = np.median(lengths[lengths >= threshold])
    mark = "  <-- берём" if threshold == MIN_CONTEXT_LENGTH else ""
    print(f"  порог {threshold:4}: останется {kept_count:5} документов, медиана {median:6.0f}{mark}")

Длина context_text по всему файлу:
  перцентиль  50:     77 символов
  перцентиль  75:    245 символов
  перцентиль  90:   1734 символов
  перцентиль 100: 105511 символов

  порог  150: останется   850 документов, медиана    829  <-- берём
  порог  300: останется   642 документов, медиана   1356
  порог  500: останется   509 документов, медиана   1825
  порог  800: останется   428 документов, медиана   2181


In [12]:
kept = [row for row in rows if len(row["context_text"]) >= MIN_CONTEXT_LENGTH]

documents = [
    {
        "document_id": row["document_id"],
        "name": row["name"],
        "city": row["city"],
        "search_text": row["search_text"],
        "context_text": row["context_text"][:MAX_CONTEXT_CHARS],
    }
    for row in kept
]

print(f"Отобрано: {len(documents)} документов")
print()

# смотрим глазами, что по ним правда можно ответить
for document in documents[:3]:
    print(f"[{document['city']}] {document['name']}")
    print(f"  поиск : {document['search_text'][:120]}")
    print(f"  ответ : {document['context_text'][:200]}...")
    print()

Отобрано: 850 документов

[Ессентуки] Lady Stretch
  поиск : Lady Stretch — объект в городе Ессентуки. Часы работы по данным карты: Mo-Su 09:00-14:00,16:00-21:00. Сайт: https://lady
  ответ : Lady Stretch — объект в городе Ессентуки. Часы работы по данным карты: Mo-Su 09:00-14:00,16:00-21:00. Сайт: https://ladystretch.com/essentuki. Телефон: +7 938 3044110....

[Ессентуки] Армянская апостольская церковь Сурб-Рипсиме
  поиск : Армянская апостольская церковь Сурб-Рипсиме — объект в городе Ессентуки. Адрес: улица Буачидзе 70. Часы работы по данным
  ответ : Армянская апостольская церковь Сурб-Рипсиме — объект в городе Ессентуки. Адрес: улица Буачидзе 70. Часы работы по данным карты: Mo-Su 08:00-19:00. Телефон: +7 988 7880368....

[Ессентуки] Грязелечебница имени Семашко
  поиск : Грязелечебница имени Семашко — достопримечательность в городе Ессентуки. Адрес: улица Семашко 10/1. Алексеевская грязеле
  ответ : Грязелечебница имени Семашко — достопримечательность в городе Ессентуки. Адрес: у

## 3. Считаем векторы

**Приставка `passage: ` обязательна.** Модель e5 обучалась так, что документ
помечается `passage: `, а вопрос — `query: `. Без приставок качество поиска
падает. В BLS для вопроса будет стоять `query: `.

`normalize_embeddings=True` приводит векторы к длине 1 — тогда скалярное
произведение и есть косинусная близость.

In [13]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL_NAME)
model.max_seq_length = INDEX_MAX_LENGTH

vectors = model.encode(
    [f"passage: {document['search_text']}" for document in documents],
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True,
)

print()
print("Матрица:", vectors.shape)
print("Длины векторов:", np.linalg.norm(vectors, axis=1)[:5], "— все по 1")

Batches: 100%|██████████| 27/27 [00:31<00:00,  1.17s/it]


Матрица: (850, 384)
Длины векторов: [1. 1. 1. 1. 1.] — все по 1


## 4. Проверяем, что поиск работает

Задаём вопрос, считаем его вектор с приставкой `query: `,
умножаем на матрицу, смотрим топ-3.

In [14]:
QUESTIONS = [
    "где в Кисловодске покататься на канатной дороге",
    "какие санатории есть в Ессентуках",
    "куда сходить с детьми в Пятигорске",
]

for question in QUESTIONS:
    query_vector = model.encode(f"query: {question}", normalize_embeddings=True)

    scores = vectors @ query_vector          # [509] — по числу на документ
    top = np.argsort(-scores)[:3]            # три наибольших

    print(f'"{question}"')
    for index in top:
        document = documents[index]
        print(f"   {scores[index]:.3f}  [{document['city']}] {document['name']}")
    print()

"где в Кисловодске покататься на канатной дороге"
   0.889  [Кисловодск] Канатная дорога
   0.859  [Кисловодск] Кисловодск: На автомобиле
   0.849  [Кисловодск] Колоннада

"какие санатории есть в Ессентуках"
   0.886  [Ессентуки] Санаторий имени И.М. Сеченова
   0.882  [Ессентуки] Курортная зона для семейного отдыха
   0.880  [Ессентуки] Ессентуки: Чем заняться

"куда сходить с детьми в Пятигорске"
   0.887  [Пятигорск] Город детей
   0.868  [Пятигорск] Детская художественная школа г. Пятигорска
   0.864  [Пятигорск] Творческая студия «Время творить»



## 5. Сохраняем

In [15]:
import json

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

(OUTPUT_DIR / "documents.json").write_text(
    json.dumps(documents, ensure_ascii=False),
    encoding="utf-8",
)
np.save(OUTPUT_DIR / "vectors.npy", vectors.astype(np.float32))

for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"{path}  {path.stat().st_size / 1024 / 1024:.2f} МБ")

model_repository\assistant_bls\1\corpus\documents.json  1.51 МБ
model_repository\assistant_bls\1\corpus\vectors.npy  1.25 МБ


In [16]:
# читаем обратно ровно так, как это сделает BLS в initialize()
check_documents = json.loads((OUTPUT_DIR / "documents.json").read_text(encoding="utf-8"))
check_vectors = np.load(OUTPUT_DIR / "vectors.npy")

print("Документов:", len(check_documents))
print("Матрица:   ", check_vectors.shape, check_vectors.dtype)
print("Совпадает: ", len(check_documents) == check_vectors.shape[0])

Документов: 850
Матрица:    (850, 384) float32
Совпадает:  True


Файлы лежат в `model_repository/assistant_bls/1/corpus/` и поедут
в контейнер вместе с BLS.